In [14]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window as w

In [2]:
df_inventory = spark.read.table("dbt_adv_pr.silver.inventory_2025_all")
df_inventory.show(20, truncate=False)

+-------------+------------+------+-----------+------------+---------+---+----------------+------------+
|snapshot_date|warehouse_id|sku_id|on_hand_qty|safety_stock|lead_time|rop|last_supply_date|holding_cost|
+-------------+------------+------+-----------+------------+---------+---+----------------+------------+
|2025-03-01   |WH01        |SKU001|119        |46          |6        |102|2025-01-24      |92.4        |
|2025-03-01   |WH01        |SKU002|194        |38          |5        |101|2025-02-16      |46.2        |
|2025-03-01   |WH01        |SKU003|0          |47          |6        |103|2025-01-24      |30.8        |
|2025-03-01   |WH01        |SKU004|319        |50          |4        |100|2025-02-19      |9.24        |
|2025-03-01   |WH01        |SKU005|30         |15          |6        |38 |2025-01-29      |26.4        |
|2025-03-01   |WH01        |SKU006|144        |24          |7        |66 |2025-02-16      |39.6        |
|2025-03-01   |WH01        |SKU007|127        |14      

In [6]:
order_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
order_df.show(20, truncate= False)

+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+
|order_id  |sku_id|item_name            |order_date|quantity|unit_price|discount_pct|amount|warehouse_id|customer_id|delivery_partner_id|order_status|dispatch_date|delivered_date|promised_date|shipping_cost|sku_rating|
+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+
|ORD0006769|SKU003|Sunflower Oil 1L     |2025-03-01|4       |215.98    |0.02        |846.64|WH02        |C00003     |DP02               |DELIVERED   |2025-03-03   |2025-03-05    |2025-03-07   |114.51       |4         |
|ORD0006769|SKU002|Whole Wheat Flour 5kg|2025-03-01|1       |306.43    |0.02        |300.3 |WH01        |C00003     |DP06   

In [10]:
inv_df = spark.read.table("dbt_adv_pr.silver.inventory_2025_all")
order_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")

daily_demand = (
    order_df
    .groupBy("warehouse_id","sku_id","order_date")
    .agg(F.sum("quantity").alias("daily_qty"))
)

demand_stats = (
    daily_demand
    .groupBy("warehouse_id","sku_id")
    .agg(
        F.avg("daily_qty").alias("avg_daily_demand"),
        F.stddev("daily_qty").alias("sigma_demand")
    )
)

Z = 1.65

result_df = (
    inv_df.join(demand_stats, on=["warehouse_id","sku_id"], how='left')
    .withColumn("sigma_demand", F.round(F.coalesce(F.col("sigma_demand"), F.lit(0.0)),2))
    .withColumn(
        "calculated_safety_stock",
        F.round(F.lit(Z) * F.col("sigma_demand") * F.sqrt(F.col("lead_time")),0)
    )
    .select(
         "snapshot_date",
        "warehouse_id",
        "sku_id",
        "lead_time",
        "sigma_demand",
        "safety_stock",              
        "calculated_safety_stock"
    )
)

result_df.show(20, truncate=False)

+-------------+------------+------+---------+------------+------------+-----------------------+
|snapshot_date|warehouse_id|sku_id|lead_time|sigma_demand|safety_stock|calculated_safety_stock|
+-------------+------------+------+---------+------------+------------+-----------------------+
|2025-03-01   |WH01        |SKU001|6        |5.94        |46          |24.0                   |
|2025-03-01   |WH01        |SKU002|5        |7.21        |38          |27.0                   |
|2025-03-01   |WH01        |SKU003|6        |4.56        |47          |18.0                   |
|2025-03-01   |WH01        |SKU004|4        |5.76        |50          |19.0                   |
|2025-03-01   |WH01        |SKU005|6        |2.88        |15          |12.0                   |
|2025-03-01   |WH01        |SKU006|7        |3.04        |24          |13.0                   |
|2025-03-01   |WH01        |SKU007|8        |2.65        |14          |12.0                   |
|2025-03-01   |WH01        |SKU008|4    

In [11]:
saftey_stock_data = result_df
saftey_stock_data.show(20, truncate=False)

+-------------+------------+------+---------+------------+------------+-----------------------+
|snapshot_date|warehouse_id|sku_id|lead_time|sigma_demand|safety_stock|calculated_safety_stock|
+-------------+------------+------+---------+------------+------------+-----------------------+
|2025-03-01   |WH01        |SKU001|6        |5.94        |46          |24.0                   |
|2025-03-01   |WH01        |SKU002|5        |7.21        |38          |27.0                   |
|2025-03-01   |WH01        |SKU003|6        |4.56        |47          |18.0                   |
|2025-03-01   |WH01        |SKU004|4        |5.76        |50          |19.0                   |
|2025-03-01   |WH01        |SKU005|6        |2.88        |15          |12.0                   |
|2025-03-01   |WH01        |SKU006|7        |3.04        |24          |13.0                   |
|2025-03-01   |WH01        |SKU007|8        |2.65        |14          |12.0                   |
|2025-03-01   |WH01        |SKU008|4    

In [12]:
saftey_stock_data.write.mode("append")\
                       .saveAsTable("dbt_adv_pr.gold.saftey_stock_data")    

In [16]:
order_df.show(20,truncate=False)

+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+
|order_id  |sku_id|item_name            |order_date|quantity|unit_price|discount_pct|amount|warehouse_id|customer_id|delivery_partner_id|order_status|dispatch_date|delivered_date|promised_date|shipping_cost|sku_rating|
+----------+------+---------------------+----------+--------+----------+------------+------+------------+-----------+-------------------+------------+-------------+--------------+-------------+-------------+----------+
|ORD0006769|SKU003|Sunflower Oil 1L     |2025-03-01|4       |215.98    |0.02        |846.64|WH02        |C00003     |DP02               |DELIVERED   |2025-03-03   |2025-03-05    |2025-03-07   |114.51       |4         |
|ORD0006769|SKU002|Whole Wheat Flour 5kg|2025-03-01|1       |306.43    |0.02        |300.3 |WH01        |C00003     |DP06   

In [23]:
date_span = order_df.select(
    (F.date_diff(F.max("order_date"),F.min("order_date")) +1).alias("days")).collect()[0]["days"]

annual_demand = (
    order_df
    .groupBy("warehouse_id","sku_id")
    .agg(
        F.round(F.sum("quantity") * (365.0 / F.lit(date_span)),2).alias("annual_demand_D")
    )
)

latest_window = w.partitionBy("warehouse_id","sku_id").orderBy(F.col("snapshot_date").desc())

latest_inv = (
    inv_df
    .withColumn("rn", F.row_number().over(latest_window))
    .filter(F.col("rn") == 1)
    .select("warehouse_id","sku_id","holding_cost")
)

S = 500.0

eoq_df = (
    annual_demand.join(latest_inv, on=["warehouse_id","sku_id"], how="inner")
    .filter(F.col("holding_cost") > 0)
    .withColumn("eoq",
                F.round(F.sqrt((2*F.col("annual_demand_D") * F.lit(S))/ F.col("holding_cost")),0))
    .orderBy("warehouse_id","sku_id")
)

eoq_df.show(20, truncate= False)

+------------+------+---------------+------------+------+
|warehouse_id|sku_id|annual_demand_D|holding_cost|eoq   |
+------------+------+---------------+------------+------+
|WH01        |SKU001|3783.09        |92.4        |202.0 |
|WH01        |SKU002|4515.11        |46.2        |313.0 |
|WH01        |SKU003|2786.91        |30.8        |301.0 |
|WH01        |SKU004|3994.83        |9.24        |658.0 |
|WH01        |SKU005|1470.08        |26.4        |236.0 |
|WH01        |SKU006|1818.95        |39.6        |214.0 |
|WH01        |SKU007|1010.3         |48.4        |144.0 |
|WH01        |SKU008|3573.37        |12.32       |539.0 |
|WH01        |SKU009|1086.93        |57.2        |138.0 |
|WH01        |SKU010|1375.3         |26.4        |228.0 |
|WH01        |SKU011|4305.39        |7.7         |748.0 |
|WH01        |SKU012|5152.35        |37.4        |371.0 |
|WH01        |SKU013|3496.74        |3.96        |940.0 |
|WH01        |SKU014|1363.2         |9.9         |371.0 |
|WH01        |

In [24]:
eoq_df.write.mode('append')\
            .saveAsTable("dbt_adv_pr.gold.eoq_data")

In [32]:
sku_rev = (
    order_df
    .groupBy("sku_id")
    .agg(
        F.round(F.sum("amount"),2).alias("total_revenue"),
        F.sum("quantity").alias("total_qty")
    )
)

rev_window = w.orderBy(F.col("total_revenue").desc())
total_rev_all = w.rowsBetween(w.unboundedPreceding,w.unboundedFollowing)

abc_df = (
    sku_rev
    .withColumn("cum_revenue",F.sum("total_revenue").over(rev_window))
    .withColumn("grand_total_rev", F.sum("total_revenue").over(total_rev_all))
    .withColumn("cum_rev_pct", F.round((F.col("cum_revenue") / F.col("grand_total_rev")) * 100,2))
    .withColumn(
        "abc_class",
        F.when(F.col("cum_rev_pct") <= 80.0,"A")
         .when(F.col("cum_rev_pct") <= 95.0, 'B')
         .otherwise("C")
    )
)

date_bounds = order_df.select(
    F.min("order_date").alias("min_dt"),
    F.max("order_date").alias("max_dt")
)

calendar_df = date_bounds.select(
    F.explode(
        F.sequence(
            F.col("min_dt").cast("date"),
            F.col("max_dt").cast("date"),
            F.expr("interval 1 day")
        )
    ).alias("order_date")
)

skus_df = order_df.select("sku_id").distinct()
sku_calendar = skus_df.crossJoin(calendar_df)

daily_sales = (
    order_df
    .groupBy("sku_id","order_date")
    .agg(F.sum("quantity").alias("daily_qty"))
)

xyz_df = (
    sku_calendar.join(daily_sales, on = ['sku_id','order_date'], how='left')
    .withColumn('daily_qty',F.coalesce(F.col("daily_qty"),F.lit(0)))
    .groupBy("sku_id")
    .agg(
        F.round(F.avg("daily_qty"),2).alias("avg_daily_qty"),
        F.round(F.stddev("daily_qty"),2).alias("stddev_daily_qty")
    )
    .withColumn("cv",
                F.when(F.col("avg_daily_qty") > 0, F.round(F.col("stddev_daily_qty")/ F.col("avg_daily_qty"),2))
                .otherwise(F.lit(999.0))
    )
    .withColumn(
        "xyz_class",
        F.when(F.col('cv') <= 0.5, "X")
         .when(F.col('cv') <= 1.0, 'Y')
         .otherwise('Z')
    )
)

abc_xyz_df = (
    abc_df.join(xyz_df, on=['sku_id'], how='inner')
    .withColumn("abc_xyz",F.concat(F.col("abc_class"),F.col("xyz_class")))
    .withColumn(
        "mover_speed",
        F.when(F.col("abc_class") == "A", "Fast Mover")
         .when(F.col("abc_class") == "B", "Medium Mover")
         .otherwise("Slower Mover")
    )
    .select(
        "sku_id",
        "total_revenue",
        "cum_rev_pct",
        "avg_daily_qty",
        "cv",
        "abc_class",
        "xyz_class",
        "abc_xyz",
        "mover_speed"
    )
    .orderBy(F.col('total_revenue').desc())
)

abc_xyz_df.show(30, truncate=False)

(
    abc_xyz_df
    .groupBy("abc_xyz","mover_speed")
    .agg(
        F.count("sku_id").alias("sku_count"),
        F.round(F.sum("total_revenue"),2).alias("segment_revenue")
    )
    .orderBy("abc_xyz")
    .show(truncate=False)
)

+------+-------------+-----------+-------------+----+---------+---------+-------+------------+
|sku_id|total_revenue|cum_rev_pct|avg_daily_qty|cv  |abc_class|xyz_class|abc_xyz|mover_speed |
+------+-------------+-----------+-------------+----+---------+---------+-------+------------+
|SKU001|2915274.07   |17.71      |28.62        |0.37|A        |X        |AX     |Fast Mover  |
|SKU012|1966278.41   |29.65      |43.41        |0.35|A        |X        |AX     |Fast Mover  |
|SKU002|1879847.06   |41.07      |34.92        |0.34|A        |X        |AX     |Fast Mover  |
|SKU003|861291.86    |46.3       |22.72        |0.4 |A        |X        |AX     |Fast Mover  |
|SKU006|670309.85    |50.37      |13.89        |0.46|A        |X        |AX     |Fast Mover  |
|SKU009|565317.51    |53.81      |9.77         |0.64|A        |Y        |AY     |Fast Mover  |
|SKU021|554597.9     |57.18      |13.17        |0.66|A        |Y        |AY     |Fast Mover  |
|SKU029|508385.7     |60.26      |23.98        |0.

In [34]:
abc_xyz_df.write.mode('append')\
                .saveAsTable('dbt_adv_pr.gold.sku_movers')

/Users/susantamukherjee/Documents/adv_log/.venv/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [39]:
annual_sales = (
    order_df
    .groupBy('warehouse_id')
    .agg(
        F.round(F.sum("quantity") * (365.0 / F.lit(date_span)),2).alias("annual_units_sold"),
        F.round(F.sum('amount') * (365.0 / F.lit(date_span)),2).alias("annual_sales_value")
    )
)

snapshot_totals = (
    inv_df
    .groupBy("warehouse_id","snapshot_date")
    .agg(F.sum("on_hand_qty").alias("snapshot_units"))
)

avg_inv = (
    snapshot_totals
    .groupBy("warehouse_id")
    .agg(F.round(F.avg('snapshot_units'),2).alias("avg_inventory_units"))
)

trunover_df = (
    annual_sales.join(avg_inv, on = 'warehouse_id', how='inner')
    .withColumn(
        "inventory_trunover_ratio",
        F.round(F.col("annual_units_sold") / F.col("avg_inventory_units"),2)
    )
    .withColumn(
        "days_inventory_outstanding",
        F.round(365 / F.col("inventory_trunover_ratio"),1)
    )
    .orderBy("warehouse_id")
)
trunover_df.show(truncate=False)

+------------+-----------------+------------------+-------------------+------------------------+--------------------------+
|warehouse_id|annual_units_sold|annual_sales_value|avg_inventory_units|inventory_trunover_ratio|days_inventory_outstanding|
+------------+-----------------+------------------+-------------------+------------------------+--------------------------+
|WH01        |66127.51         |1.141225231E7     |3349.83            |19.74                   |18.5                      |
|WH02        |51106.05         |8720217.7         |2688.01            |19.01                   |19.2                      |
|WH03        |44104.5          |7529822.03        |2322.39            |18.99                   |19.2                      |
|WH04        |16759.75         |2891280.07        |1181.28            |14.19                   |25.7                      |
|WH05        |15448.98         |2582589.62        |1089.44            |14.18                   |25.7                      |
+-------

In [40]:
trunover_df.write.mode('append')\
                 .saveAsTable('dbt_adv_pr.gold.warehouse_trunover')
